In [143]:
import requests
from bs4 import BeautifulSoup

import re
import pandas as pd
from rapidfuzz import fuzz

import asyncio

# uv add playwright
# uv run playwright install
# uv run playwright install-deps
from playwright.async_api import async_playwright 

In [271]:
def scrape_autokinito(base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):   
    # Set target page
    url = f"{base}/antiprosopies/"

    # Get the page's HTML
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    return soup


def get_aftokinito_cars(soup, base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):
    # Car brands and models are located under the "make/" URL path
    make_links = []

    # Find all <a> tags where the href contains the /make/ path
    for a in soup.select("a[href*='/make/']"):
        make_links.append(base + a["href"])

    cars = []

    # Loop through each brand link
    for link in set(make_links):
        r = requests.get(link, headers=HEADERS)
        s = BeautifulSoup(r.text, "html.parser")

        # Extract brand name
        brand = link.rstrip("/").split("/")[-1].upper()

        # Extract models
        for m in s.select("h2, h3"):
            model = m.get_text(strip=True)
            if model:
                cars.append((brand, model))
    
    return cars
    

def validate_aftokinito_cars(cars):
    valid_models = dict()

    for car in cars:
        brand = car[0].strip() #.lower().strip()
        model = car[1].strip() #.lower().strip()
        
        if brand not in valid_models:
            valid_models[brand] = []
        
        # In aftokinito's site, each model starts with its brand, but the way brand is stored in model may slightly differ
        model_prefix = model[:len(brand)]
        similarity = fuzz.ratio(brand, model_prefix)
        
        if similarity >= 70:
            valid_models[brand].append(model)

    return valid_models


def convert_aftokinito_to_pandas_and_clean(cars):
    rows = [(brand, model) for brand, models in cars.items() for model in models]
    df = pd.DataFrame(rows, columns=['Brand', 'Model'])

    df = df.drop_duplicates()

    # Remove rows where 'Model' contains Greek characters: these are adverts
    df = df[~df['Model'].str.contains(r'[α-ωΑ-Ω]', regex=True)]

    # Remove brand prefix from Model for every brand 
    df["Model"] = (
        df["Model"]
        .str.replace(
            r'^(' + '|'.join(df["Brand"].unique()) + r')\s+',
            '',
            regex=True
        )
    )
    df["Source_URL"] = "https://autokinito.com.cy/antiprosopies/"

    return df.reset_index(drop=True)

In [272]:
async def scrape_bazaraki(pages=1):
    base_url = "https://www.bazaraki.com/car-motorbikes-boats-and-parts/cars-trucks-and-vans/"
    soups = []
    
    # Blocked by Cloudflare: https://scrapeops.io/web-scraping-playbook/how-to-bypass-cloudflare/ 
    # Bypass using playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # important for Cloudflare
            args=["--disable-blink-features=AutomationControlled"]
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 900},
            locale="en-US"
        )

        page = await context.new_page()

        for i in range(1, pages + 1):
            url = f"{base_url}?page={i}"

            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # wait for page structure 
            await page.wait_for_selector("body")

            await page.wait_for_timeout(3000)  # allow JS rendering

            html = await page.content()
            soups.append(BeautifulSoup(html, "html.parser"))

        await browser.close()

    return soups


def get_bazaraki_cars(soups):
    cars = []
    for soup in soups:

        # Ads in div.advert; brand + model in <a.advert__content-title>
        for advert in soup.select("div.advert"):

            title_tag = advert.select_one("a.advert__content-title")
            if not title_tag:
                continue

            title = title_tag.get_text(strip=True)
            cars.append(title)

    return cars


def convert_bazaraki_to_pandas_and_clean(cars):
    # Remove engine sizes and year
    cars = [
        re.sub(r"\s\d{4}$", "",
        re.sub(r"\s\d,\dL", "", car))
        for car in cars
    ]

    brand_model_split = [car.split() for car in cars]
    df = pd.DataFrame(brand_model_split) # columns are numbered 0, 1, 2…

    df = df.drop_duplicates()
    df = df[df[0] != "Cars"]

    # In some cars, parts of the brand appear in model because the brand is split across multiple words.
    # For example, "Land Rover" gets "Land" as the brand and "Rover" as the model.
    # This is to verify that brands and models are captured correctly
    # print(df[0].unique())
    idx = df.index[
        ((df[0] == "Land") & (df[1] == "Rover")) |
        ((df[0] == "Alfa") & (df[1] == "Romeo")) |
        ((df[0] == "Rolls") & (df[1] == "Royce")) |
        ((df[0] == "Aston") & (df[1] == "Martin"))
    ]

    # Combine columns 0 and 1 into 0
    df.loc[idx, 0] = df.loc[idx, 0] + ' ' + df.loc[idx, 1]

    # Combine remaining columns 2,3,4 into 1, ignoring NaNs
    df.loc[idx, 1] = df.loc[idx, [2, 3, 4]].fillna("").agg(' '.join, axis=1).str.strip()

    # Clear original 2,3,4 columns
    df.loc[idx, [2, 3, 4]] = pd.NA

    # Combine 1,2,3,4 columns into 1 for the remaining records
    df.loc[:, 1] =  df.loc[:, [1,2,3,4]].fillna("").agg(' '.join, axis=1).str.strip()

    # Mark 0 as Brand and 1 as Model
    df = df[[0,1]].rename(columns={0: 'Brand', 1: 'Model'})
    df["Source_URL"] = "https://www.bazaraki.com/car-motorbikes-boats-and-parts/cars-trucks-and-vans/"

    return df.reset_index(drop=True)

In [274]:
# aftokinito_soup = scrape_autokinito()
# aftokinito_cars = get_aftokinito_cars(aftokinito_soup)
valid_aftokinito_cars = validate_aftokinito_cars(aftokinito_cars)
aftokinito = convert_aftokinito_to_pandas_and_clean(valid_aftokinito_cars)

# bazaraki_soups = await scrape_bazaraki(175)
# bazaraki_cars = get_bazaraki_cars(bazaraki_soups)
bazaraki = convert_bazaraki_to_pandas_and_clean(bazaraki_cars)



# Rapid fuzz to identify existing models

In [275]:
print(len(aftokinito))
aftokinito

370


,Brand,Model,Source_URL
0,BYD,Dolphin Surf,https://autokinito.com.cy/antiprosopies/
1,BYD,Atto 3,https://autokinito.com.cy/antiprosopies/
2,BYD,Sealion 5 DM-i,https://autokinito.com.cy/antiprosopies/
3,BYD,Seal U DM-I,https://autokinito.com.cy/antiprosopies/
4,BYD,Sealion 7,https://autokinito.com.cy/antiprosopies/
5,BYD,Seal,https://autokinito.com.cy/antiprosopies/
6,CITROEN,C3,https://autokinito.com.cy/antiprosopies/
7,CITROEN,C3 Aircross,https://autokinito.com.cy/antiprosopies/
8,CITROEN,C5 Aircross,https://autokinito.com.cy/antiprosopies/
9,CITROEN,e-C3,https://autokinito.com.cy/antiprosopies/
